In [ ]:
def is_iterable(data):
    return isinstance(data, list)

def flatten(data):
    if not is_iterable(data):
        return [data]

    out = []
    for item in data:
        out.extend(flatten(item))
    return out

def get_shape(data):
    if not is_iterable(data):
        return ()

    assert data, "empty tensors skipped for now"

    inner = get_shape(data[0])
    
    for item in data:
        assert get_shape(item) == inner, "ragged tensor"

    return (len(data),) + inner

def get_strides_from_shape(shape):
    strides = []
    running = 1
    for size in reversed(shape):
        strides.append(running)
        running *= size
    return tuple(reversed(strides))

def index_to_position(index, strides):
    return sum(i * stride for i, stride in zip(index * strides))

def broadcast_shape(a_shape: tuple[int, ...], b_shape: tuple[int, ...]):
    output_shape = []

    for i in range(max(len(a_shape), len(b_shape))):
        a = a_shape[-1 - i] if i < len(a_shape) else 1
        b = b_shape[-1 - i] if i < len(b_shape) else 1

        assert a == b or a == 1 or b == 1, f"cannot broadcast {a_shape} and {b_shape}"

        output_shape.append(max(a, b))

    return tuple(reversed(output_shape))

def prod(values):
    output = 1
    for value in values:
        output *= value
    return output

def normalize_index(index, ndim):
    if not isinstance(index, tuple):
        index = (index,)

    if sum(x is Ellipsis for x in index) > 1:
        raise IndexError("only one ellipsis allowed")

    if Ellipsis in index:
        used = sum(x is not Ellipsis for x in index)
        fill = ndim - used
        if fill < 0:
            raise IndexError("too many indices")

        out = []
        for x in index:
            if x is Ellipsis:
                out.extend([slice(None)] * fill)
            else:
                out.append(x)
        index = tuple(out)

    if len(index) > ndim:
        raise IndexError("too many indices")

    return index + (slice(None),) * (ndim - len(index))

class Tensor:

    def __init__(
        self,
        data,
        requires_grad: bool,
        children: tuple[Tensor] = (),
    ):
        self._requires_grad = requires_grad

        self._storage = flatten(data)
        self._shape = get_shape(data)
        self._strides = get_strides_from_shape(self._shape)

        self._grad = [0.0 for _ in data]
        self._children = children

    def __getitem__(self, index):
        index = normalize_index(index, ndim=len(self.shape))


    def __mul__(self, other) -> Tensor:
        raise NotImplementedError
    
    @classmethod
    def matmul(A: Tensor, B: Tensor) -> Tensor:
        *a_batch, m, n = A.shape
        *b_batch, n_b, p = B.shape

        assert n == n_b

        output_shape = broadcast_shape(A.shape, B.shape)

        output = Tensor([0.0] * prod(output_shape))

        for output_index in range(len(output)):
            total = 0.0

            for k in range(n):
                total += (
                    A[..., m, k] * B[..., k, p]
                )

            output[..., m, p] = total

        return output

    
    def __matmul__(self, other) -> Tensor:
        *a_batch, m, n = self.shape
        *b_batch, n_b, p = other.shape

        assert n == n_b

        output_shape = broadcast_shape(self.shape, other.shape)

        output = Tensor([0.0] * prod(output_shape))

        for output_index in range(len(output)):
            total = 0.0

            for k in range(n):
                total += (
                    self[..., m, k] * other[..., k, p]
                )

        return output

    def size(self):
        return self._shape
    
    @property
    def shape(self):
        return self.size()
    
    def sum(self) -> Tensor:
        raise NotImplementedError

    # and a bunch of other operations like addition and subtraction etc

    def backward(self) -> None:
        raise NotImplementedError

In [ ]:
x = Tensor([[1, 2, 3],
            [4, 5, 6]], requires_grad=True)   # shape (2, 3)

w = Tensor([10, 20, 30], requires_grad=True)   # shape (3,)

y = x * w                                     # broadcast w -> shape (2, 3)
loss = y.sum()                                # scalar
loss.backward()